# BingePlay — SQL Minor Project
**The Unlox Academy**

This notebook connects to the `bingeplay` MySQL database and answers each project question with a SQL query executed via `pandas.read_sql`.

In [4]:
pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Users\cjaye\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
pip install sqlalchemy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Users\cjaye\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus


# --- Update these with your actual MySQL credentials ---
USER = "root"
PASSWORD = quote_plus("Govijat@03")
HOST = "localhost"       # or your DB host
PORT = 3306
DATABASE = "bingeplay"

engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}")

# Quick connection test
pd.read_sql("SELECT 1 AS connected;", engine)

,connected
0,1


## Q1 — Active subscriptions & monthly revenue
**Answer: active subscriptions = \_\_\_, monthly revenue (INR) = \_\_\_**

In [18]:
query = """
SELECT COUNT(*) AS active_subscriptions,
       SUM(monthly_price_inr) AS monthly_revenue
FROM subscriptions
WHERE status = 'active'
AND (end_date IS NULL OR end_date > '2024-06-30');
"""
df = pd.read_sql(query, engine)
print(df)


DatabaseError: Execution failed on sql '
SELECT COUNT(*) AS active_subscriptions,
       SUM(monthly_price_inr) AS monthly_revenue
FROM subscriptions
WHERE status = 'active'
AND (end_date IS NULL OR end_date > '2024-06-30');
': (pymysql.err.OperationalError) (1054, "Unknown column 'status' in 'where clause'")
[SQL: 
SELECT COUNT(*) AS active_subscriptions,
       SUM(monthly_price_inr) AS monthly_revenue
FROM subscriptions
WHERE status = 'active'
AND (end_date IS NULL OR end_date > '2024-06-30');
]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

## Q2 — Signup momentum
**Answer: peak signup month = \_\_\_ (\_\_\_ signups)**

In [ ]:
query = """
SELECT MONTH(signup_date) AS month, MONTHNAME(signup_date) AS month_name,
       COUNT(*) AS signup_count
FROM users
WHERE YEAR(signup_date) = 2024
GROUP BY MONTH(signup_date), MONTHNAME(signup_date)
ORDER BY month;
"""
df = pd.read_sql(query, engine)
print(df)


   month month_name  signup_count
0      1    January           350
1      2   February           400
2      3      March           500
3      4      April           550
4      5        May           600
5      6       June           600


## Q3 — Device analytics
**Answer: top device by sessions = \_\_\_, highest completion rate = \_\_\_**


In [10]:
query = """
SELECT device_type, COUNT(*) AS total_sessions,
       SUM(watch_minutes) AS total_watch_minutes,
       ROUND(AVG(watch_minutes), 2) AS avg_watch_minutes,
       ROUND(100.0 * SUM(completed) / COUNT(*), 2) AS completion_rate_pct
FROM watch_sessions
WHERE user_id IS NOT NULL
GROUP BY device_type
ORDER BY total_sessions DESC;
"""
df = pd.read_sql(query, engine)
print(df)


  device_type  total_sessions  total_watch_minutes  avg_watch_minutes  \
0      Mobile           50172            1504355.0              29.98   
1          TV           27981             840595.0              30.04   
2      Laptop           15105             453434.0              30.02   
3      Tablet            7091             210733.0              29.72   

   completion_rate_pct  
0                60.24  
1                59.98  
2                60.51  
3                59.79  


## Q4 — Rating distribution
**Answer: % of ratings that are 4 or 5 stars = \_\_\_%**

In [ ]:
query = """
SELECT stars, COUNT(*) AS rating_count,
ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM ratings), 2) AS percentage
FROM ratings
GROUP BY stars
ORDER BY stars;
"""
df = pd.read_sql(query, engine)
print(df)

query_pct = """
SELECT ROUND(100.0 * SUM(CASE WHEN stars IN (4,5) THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_4_or_5_stars
FROM ratings;
"""
df_pct = pd.read_sql(query_pct, engine)
print(df_pct)

   stars  rating_count  percentage
0      1           234        4.68
1      2           352        7.04
2      3           847       16.94
3      4          1781       35.62
4      5          1786       35.72
   pct_4_or_5_stars
0             71.34


## Q5 — Originals vs. acquired
**Answer: originals avg IMDB rating = \_\_\_ vs acquired avg IMDB rating = \_\_\_**

In [19]:
query = """
SELECT is_original, COUNT(*) AS num_shows,
ROUND(AVG(imdb_rating), 2) AS avg_imdb_rating,
ROUND(AVG(release_year), 2) AS avg_release_year
FROM shows
GROUP BY is_original
ORDER BY is_original DESC;
"""
df = pd.read_sql(query, engine)
print(df)


   is_original  num_shows  avg_imdb_rating  avg_release_year
0            1         30             7.92           2020.37
1            0         70             6.63           2020.73


## Q6 — Binge day detection
**Answer: total binge days = \_\_\_, user with most binge days = \_\_\_ (\_\_\_ days)**

In [ ]:
query_total = """
WITH daily_show_counts AS (
SELECT user_id, show_id, session_date, COUNT(*) AS sessions_that_day
FROM watch_sessions
WHERE user_id IS NOT NULL
AND session_date BETWEEN '2024-04-01' AND '2024-06-30'
GROUP BY user_id, show_id, session_date
HAVING COUNT(*) >= 5)
SELECT COUNT(*) AS total_binge_days FROM daily_show_counts;
"""
print(pd.read_sql(query_total, engine))

query_top_user = """
WITH daily_show_counts AS (
SELECT user_id, show_id, session_date, COUNT(*) AS sessions_that_day
FROM watch_sessions
WHERE user_id IS NOT NULL
AND session_date BETWEEN '2024-04-01' AND '2024-06-30'
GROUP BY user_id, show_id, session_date
HAVING COUNT(*) >= 5)
SELECT user_id, COUNT(*) AS binge_day_count
FROM daily_show_counts
GROUP BY user_id
ORDER BY binge_day_count DESC
LIMIT 1;
"""
print(pd.read_sql(query_top_user, engine))


   total_binge_days
0               414
  user_id  binge_day_count
0  U02956                8


## Q7 — Q1 signups who never watched (the NULL trap)
**Answer: total Q1 2024 signups = \_\_\_, never-watched count = \_\_\_**

In [ ]:
query_total = """
SELECT COUNT(*) AS total_q1_signups FROM users
WHERE signup_date BETWEEN '2024-01-01' AND '2024-03-31';
"""
print(pd.read_sql(query_total, engine))

query_never_watched = """
SELECT COUNT(*) AS never_watched_count FROM users u
WHERE u.signup_date BETWEEN '2024-01-01' AND '2024-03-31'
AND NOT EXISTS (
SELECT 1 FROM watch_sessions ws WHERE ws.user_id = u.user_id);
"""
print(pd.read_sql(query_never_watched, engine))

   total_q1_signups
0              1250
   never_watched_count
0                  226


## Q8 — Over-paying Premium/Family users
**Answer: overpaying users = \_\_\_**

In [ ]:
query = """
WITH ranked_subs AS ( SELECT s.user_id, s.plan, s.start_date,
ROW_NUMBER() OVER (PARTITION BY s.user_id ORDER BY s.start_date DESC) AS rn FROM subscriptions s
WHERE s.start_date <= '2024-06-30'
AND (s.end_date IS NULL OR s.end_date >= '2024-06-30')),
current_plan AS (SELECT user_id, plan FROM ranked_subs WHERE rn = 1),
user_show_tiers AS (
SELECT ws.user_id,
MIN(sh.min_plan = 'Basic') AS all_basic  -- 0 if ANY non-basic show watched
FROM watch_sessions ws
JOIN shows sh ON sh.show_id = ws.show_id
WHERE ws.user_id IS NOT NULL
GROUP BY ws.user_id)
SELECT COUNT(*) AS overpaying_users FROM current_plan cp
JOIN user_show_tiers ust ON ust.user_id = cp.user_id
WHERE cp.plan IN ('Premium','Family')
AND ust.all_basic = 1;
"""
df = pd.read_sql(query, engine)
print(df)


   overpaying_users
0                 8


## Q9 — Upgrade success cohort
**Answer: number of users who upgraded = \_\_\_, avg days signup → first upgrade = \_\_\_**

In [ ]:
query = """
WITH first_plan AS ( SELECT user_id, plan, start_date,
ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY start_date) AS rn FROM subscriptions),
first_upgrade AS ( SELECT s.user_id, MIN(s.start_date) AS upgrade_date FROM subscriptions s
JOIN first_plan fp ON s.user_id = fp.user_id WHERE fp.rn = 1
AND fp.plan = 'Basic'
AND s.start_date > fp.start_date
AND s.plan IN ('Premium', 'Family')
GROUP BY s.user_id),
active_users AS ( SELECT DISTINCT user_id FROM subscriptions
WHERE start_date <= '2024-06-30'
AND (end_date IS NULL OR end_date >= '2024-06-30'))
SELECT COUNT(*) AS number_of_users,
ROUND(AVG(DATEDIFF(fu.upgrade_date, u.signup_date)), 2) AS avg_days_signup_to_first_upgrade FROM users u
JOIN first_plan fp ON u.user_id = fp.user_id AND fp.rn = 1
JOIN first_upgrade fu ON u.user_id = fu.user_id
JOIN active_users au ON u.user_id = au.user_id
WHERE u.signup_date BETWEEN '2024-01-01' AND '2024-01-31'
AND fp.plan = 'Basic';
"""
df = pd.read_sql(query, engine)
print(df)


   number_of_users  avg_days_signup_to_first_upgrade
0               55                             64.96


## Q10 — Cliffhanger comebacks
**Answer: total comeback events = \_\_\_, top show = \_\_\_ (\_\_\_ comebacks)**

In [ ]:
query = """
WITH comeback_events AS ( SELECT ws1.user_id, ws1.show_id, ws1.session_date AS incomplete_date FROM watch_sessions ws1
WHERE ws1.completed = 0
AND EXISTS (SELECT 1 FROM watch_sessions ws2
WHERE ws2.user_id = ws1.user_id
AND ws2.show_id = ws1.show_id
AND ws2.session_date > ws1.session_date
AND DATEDIFF(ws2.session_date, ws1.session_date) BETWEEN 1 AND 7)),
show_comebacks AS (
SELECT show_id, COUNT(*) AS comeback_count
FROM comeback_events
GROUP BY show_id),
top_show AS ( SELECT show_id, comeback_count FROM show_comebacks
ORDER BY comeback_count DESC
LIMIT 1)
SELECT (SELECT COUNT(*) FROM comeback_events) AS total_cliffhanger_comeback_events,
ts.show_id, s.title
FROM top_show ts
JOIN shows s ON ts.show_id = s.show_id;
"""
df = pd.read_sql(query, engine)
print(df)


   total_cliffhanger_comeback_events show_id             title
0                               4420    S088  Rayalaseema Raga


## Q11 — Consecutive-week engagement
**Answer: users with 4+ week streak = \_\_\_, longest streak = \_\_\_ weeks (user \_\_\_)**

In [ ]:
query = """
WITH user_weeks AS (SELECT DISTINCT user_id, YEARWEEK(session_date, 3) AS iso_week FROM watch_sessions
WHERE user_id IS NOT NULL),
numbered_weeks AS ( SELECT user_id, iso_week,
ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY iso_week) AS rn FROM user_weeks),
week_groups AS ( SELECT user_id, iso_week, iso_week - rn AS grp FROM numbered_weeks),
streaks AS ( SELECT user_id, COUNT(*) AS streak_length FROM week_groups
GROUP BY user_id, grp),
longest_streak AS ( SELECT user_id, streak_length FROM streaks
ORDER BY streak_length DESC
LIMIT 1)
SELECT
(SELECT COUNT(DISTINCT user_id) FROM streaks WHERE streak_length >= 4) AS users_with_4plus_streak,
(SELECT streak_length FROM longest_streak) AS longest_streak_weeks,
(SELECT user_id FROM longest_streak) AS user_id_with_longest_streak;
"""
df = pd.read_sql(query, engine)
print(df)


   users_with_4plus_streak  longest_streak_weeks user_id_with_longest_streak
0                     1675                    26                      U01658


## Q12 — Churn signal detection
**Answer: total users flagged as churn risks = \_\_\_**

In [ ]:
query = """
WITH monthly_watch AS ( SELECT user_id,
SUM(CASE WHEN session_date BETWEEN '2024-05-01' AND '2024-05-31' THEN watch_minutes ELSE 0 END) AS may_watch_minutes,
SUM(CASE WHEN session_date BETWEEN '2024-06-01' AND '2024-06-30' THEN watch_minutes ELSE 0 END) AS june_watch_minutes
FROM watch_sessions
WHERE user_id IS NOT NULL
GROUP BY user_id),
churn_users AS (
SELECT mw.user_id, u.name, mw.may_watch_minutes, mw.june_watch_minutes,
ROUND(((mw.may_watch_minutes - mw.june_watch_minutes) / mw.may_watch_minutes) * 100, 2) AS drop_percentage
FROM monthly_watch mw
JOIN users u ON mw.user_id = u.user_id
WHERE mw.may_watch_minutes > 0
AND mw.june_watch_minutes <= mw.may_watch_minutes * 0.5)
SELECT user_id, name, may_watch_minutes, june_watch_minutes, drop_percentage,
(SELECT COUNT(*) FROM churn_users) AS total_churn_signal_users
FROM churn_users
ORDER BY drop_percentage DESC, user_id;
"""
df = pd.read_sql(query, engine)
print(df.head(20))  # preview first 20 rows; total_churn_signal_users column has the full count


   user_id              name  may_watch_minutes  june_watch_minutes  \
0   U00023    Amit Mukherjee               43.0                 0.0   
1   U00166  Shaurya Malhotra               94.0                 0.0   
2   U00211        Ravi Menon              336.0                 0.0   
3   U00225     Kritika Patil              209.0                 0.0   
4   U00237    Shaurya Bansal              126.0                 0.0   
5   U00262        Suresh Roy              158.0                 0.0   
6   U00271       Krish Patel              227.0                 0.0   
7   U00282        Tara Kumar               47.0                 0.0   
8   U00289    Rohan Krishnan               18.0                 0.0   
9   U00292    Kavitha Kamath              594.0                 0.0   
10  U00371      Prisha Patil               93.0                 0.0   
11  U00426    Aarav Banerjee               80.0                 0.0   
12  U00438   Mahesh Krishnan              101.0                 0.0   
13  U0